# TinyCLIP efficiency: MHA -> GQA -> MQA

Convert a pretrained TinyCLIP's attention to grouped/multi-query K/V (both encoders), uptrain briefly to recover accuracy, and compare zero-shot accuracy vs cost.

In [ ]:
import os, sys, copy, json
import torch
root = os.getcwd()
while root != '/' and not os.path.isdir(os.path.join(root, 'gqa_study')):
    root = os.path.dirname(root)
sys.path.insert(0, root)
from gqa_study import models, variants, bench, recover, data
from gqa_study import eval as zeroshot
device = 'cuda' if torch.cuda.is_available() else 'cpu'
cuda = device == 'cuda'
RESULTS_DIR, DATA_DIR = 'results', 'data'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
print(device, torch.__version__)

## 1. Load pretrained model + sanity check

In [ ]:
MODEL_KEY = 'tinyclip-8m'
spec = models.MODELS[MODEL_KEY]
orig, preprocess, tokenizer = models.load_tinyclip(MODEL_KEY, device)
orig = orig.float().eval()
info = models.describe_attention(orig)
N_HEADS = info['num_heads']
assert info['num_heads'] == spec.n_heads and info['num_blocks'] == spec.layers
print(info)

In [ ]:
@torch.no_grad()
def make_variant(num_kv_heads):
    m = copy.deepcopy(orig)
    variants.convert(m, num_kv_heads)
    return m.to(device).float().eval()

img = torch.randn(8, 3, spec.image_size, spec.image_size, device=device)
txt = tokenizer(['a photo of a cat', 'a dog']).to(device)
mha = make_variant(N_HEADS)
with torch.no_grad():
    di = (orig.encode_image(img) - mha.encode_image(img)).abs().max().item()
    dt = (orig.encode_text(txt) - mha.encode_text(txt)).abs().max().item()
print('image diff', di, 'text diff', dt)
assert di < 1e-4 and dt < 1e-4
del mha

## 2. Zero-shot after conversion (no uptraining)

In [ ]:
METHODS = models.method_grid(N_HEADS)
DATASETS = ['cifar10', 'cifar100']
loaders = data.build_loaders(DATASETS, DATA_DIR, preprocess, batch_size=128)
print(METHODS)

In [ ]:
def eval_variant(mdl):
    row = {}
    for ds in loaders:
        clf = zeroshot.build_text_classifier(mdl, tokenizer, data.CLASSNAMES[ds], data.TEMPLATES[ds], device)
        row[ds] = zeroshot.evaluate_classifier(mdl, clf, loaders[ds], device)
    return row

converted = {}
for name, kv in METHODS:
    mdl = make_variant(kv)
    converted[name] = eval_variant(mdl)
    for ds in loaders:
        print(f"{name:6s} {ds:9s} top1={converted[name][ds]['top1']*100:5.2f}")
    del mdl
    if cuda: torch.cuda.empty_cache()
json.dump(converted, open(f'{RESULTS_DIR}/converted_{MODEL_KEY}.json', 'w'), indent=1)

## 3. Uptrain (distill both encoders' attention back toward the MHA teacher)

In [ ]:
UPTRAIN_STEPS = 300
teacher = copy.deepcopy(orig).to(device).eval()
for p in teacher.parameters(): p.requires_grad_(False)
distill_set = data.build_dataset('cifar100', DATA_DIR, preprocess)
distill_loader = torch.utils.data.DataLoader(distill_set, batch_size=32, shuffle=True, num_workers=2)
distill_texts = tokenizer([f'a photo of a {c}.' for c in data.CLASSNAMES['cifar100']])

In [ ]:
uptrained = {}
for name, kv in METHODS:
    if name == 'MHA':
        uptrained[name] = converted[name]
        continue
    print(name)
    mdl = recover.uptrain(make_variant(kv), teacher, distill_loader, distill_texts, device, steps=UPTRAIN_STEPS)
    uptrained[name] = eval_variant(mdl)
    for ds in loaders:
        print(f"  {ds:9s} top1={uptrained[name][ds]['top1']*100:5.2f}")
    del mdl
    if cuda: torch.cuda.empty_cache()
json.dump(uptrained, open(f'{RESULTS_DIR}/uptrained_{MODEL_KEY}.json', 'w'), indent=1)

## 4. Efficiency benchmark (params, latency, peak memory)

In [ ]:
bn = {}
for name, kv in METHODS:
    mdl = make_variant(kv)
    if cuda: mdl = mdl.half()
    rep = variants.param_report(mdl)
    t = bench.benchmark(bench.ImageTower(mdl), (64, 3, spec.image_size, spec.image_size), device)
    bn[name] = {**t, **rep}
    print(f"{name:6s} p50={t['p50_ms']:6.2f}ms thr={t['throughput_img_s']:7.1f} mem={t['peak_mem_mb']} kv={rep['kv']:,} total={rep['total']:,}")
    del mdl
    if cuda: torch.cuda.empty_cache()
json.dump(bn, open(f'{RESULTS_DIR}/bench_{MODEL_KEY}.json', 'w'), indent=1)

## 5. Results table

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 13, 'axes.titlesize': 15, 'axes.titleweight': 'bold',
                     'figure.dpi': 110, 'savefig.dpi': 200, 'savefig.bbox': 'tight'})
def avg(row): return sum(row[d]['top1'] for d in DATASETS) / len(DATASETS) * 100
rows = []
for name, kv in METHODS:
    b = bn[name]
    rows.append({'method': name, 'kv_heads': kv,
                 'top1_converted': round(avg(converted[name]), 2),
                 'top1_uptrained': round(avg(uptrained[name]), 2),
                 **{d + '_uptrained': round(uptrained[name][d]['top1']*100, 2) for d in DATASETS},
                 'kv_params': b['kv'], 'total_params': b['total'],
                 'p50_ms': round(b['p50_ms'], 2), 'throughput_img_s': round(b['throughput_img_s'], 1),
                 'peak_mem_mb': b['peak_mem_mb']})
df = pd.DataFrame(rows)
df.to_csv(f'{RESULTS_DIR}/summary_{MODEL_KEY}.csv', index=False)
M = list(df['method'])
COLORS = ['#4C72B0', '#DD8452', '#55A868', '#C44E52'][:len(M)]
def save(fig, tag): fig.savefig(f'{RESULTS_DIR}/{tag}_{MODEL_KEY}.png'); return fig
df

## 6. Poster visualizations

### Accuracy: converted vs uptrained

In [ ]:
conv = [df[df.method == m]['top1_converted'].values[0] for m in M]
up = [df[df.method == m]['top1_uptrained'].values[0] for m in M]
x = range(len(M)); w = 0.38
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.bar([xi - w/2 for xi in x], conv, w, label='after conversion')
ax.bar([xi + w/2 for xi in x], up, w, label='after uptraining')
ax.axhline(up[0], color='gray', ls='--', lw=1, label='MHA baseline')
ax.set_xticks(list(x)); ax.set_xticklabels(M)
ax.set_ylabel('avg zero-shot top-1 (%)'); ax.set_title('Accuracy recovered by uptraining')
ax.legend(); ax.grid(True, axis='y', alpha=0.3)
save(fig, 'accuracy_recovery'); plt.show()

### Accuracy per dataset (uptrained)

In [ ]:
x = range(len(M)); w = 0.8 / (len(DATASETS) + 1)
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for i, ds in enumerate(DATASETS):
    vals = [df[df.method == m][ds + '_uptrained'].values[0] for m in M]
    ax.bar([xi + i * w for xi in x], vals, w, label=ds)
av = [df[df.method == m]['top1_uptrained'].values[0] for m in M]
ax.bar([xi + len(DATASETS) * w for xi in x], av, w, label='avg', color='#333333')
ax.set_xticks([xi + w * len(DATASETS) / 2 for xi in x]); ax.set_xticklabels(M)
ax.set_ylabel('zero-shot top-1 (%)'); ax.set_title('Uptrained accuracy by dataset')
ax.legend(); ax.grid(True, axis='y', alpha=0.3)
save(fig, 'accuracy_bars'); plt.show()

### K/V parameter compression

In [ ]:
kvp = [df[df.method == m]['kv_params'].values[0] for m in M]
base = kvp[0]
fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(M, [k / 1e6 for k in kvp], color=COLORS)
for r, k in zip(bars, kvp):
    ax.annotate(f'{100*k/base:.0f}%', (r.get_x() + r.get_width()/2, r.get_height()), ha='center', va='bottom')
ax.set_ylabel('K/V parameters (M)'); ax.set_title('K/V projection size (vs MHA)')
ax.grid(True, axis='y', alpha=0.3)
save(fig, 'kv_compression'); plt.show()

### Throughput and peak memory

In [ ]:
thr = [df[df.method == m]['throughput_img_s'].values[0] for m in M]
mem = [df[df.method == m]['peak_mem_mb'].values[0] for m in M]
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].bar(M, thr, color=COLORS); ax[0].set_ylabel('images / sec'); ax[0].set_title('Image-encoder throughput')
if all(v is not None for v in mem):
    ax[1].bar(M, mem, color=COLORS); ax[1].set_ylabel('peak memory (MB)'); ax[1].set_title('Peak GPU memory')
else:
    ax[1].set_title('Peak GPU memory (GPU only)')
for a in ax: a.grid(True, axis='y', alpha=0.3)
fig.tight_layout(); save(fig, 'throughput_memory'); plt.show()

### Accuracy vs cost trade-off (uptrained)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
for a, xcol, xlab in [(ax[0], 'p50_ms', 'image encoder p50 latency (ms)'), (ax[1], 'kv_params', 'K/V parameters')]:
    for m, c in zip(M, COLORS):
        r = df[df.method == m].iloc[0]
        a.scatter(r[xcol], r['top1_uptrained'], s=120, color=c, zorder=3)
        a.annotate(m, (r[xcol], r['top1_uptrained']), textcoords='offset points', xytext=(6, 6))
    a.set_xlabel(xlab); a.set_ylabel('avg zero-shot top-1 (%)'); a.grid(True, alpha=0.3)
ax[0].set_title('Accuracy vs speed'); ax[1].set_title('Accuracy vs K/V size')
fig.tight_layout(); save(fig, 'tradeoff'); plt.show()

### Relative to MHA baseline (uptrained)

In [ ]:
acc = [df[df.method == m]['top1_uptrained'].values[0] for m in M]
rel = {'accuracy retained': [a / acc[0] * 100 for a in acc],
       'K/V params': [k / kvp[0] * 100 for k in kvp],
       'throughput': [t / thr[0] * 100 for t in thr]}
x = range(len(M)); w = 0.8 / len(rel)
fig, ax = plt.subplots(figsize=(8, 4.5))
for i, (lab, vals) in enumerate(rel.items()):
    ax.bar([xi + i * w for xi in x], vals, w, label=lab)
ax.axhline(100, color='gray', ls='--', lw=1)
ax.set_xticks([xi + w for xi in x]); ax.set_xticklabels(M)
ax.set_ylabel('% of MHA'); ax.set_title('Each variant relative to MHA')
ax.legend(); ax.grid(True, axis='y', alpha=0.3)
save(fig, 'relative_to_mha'); plt.show()